# Librosa 音频频域分析教程

本教程展示如何使用 Librosa 库加载音频文件并获取双声道的频域信息。

## 功能概述
- 加载音频文件（支持单声道和双声道）
- 使用 STFT（短时傅里叶变换）获取频域信息
- 分离左右声道并分别分析
- 可视化频谱图

In [33]:
# 导入必要的库
# 检查NumPy GPU加速
try:
    import cupy as np
    print(f"CuPy (GPU) 可用: {np.cuda.runtime.runtimeGetVersion()}")
    print(f"GPU设备: {np.cuda.Device(0)}")
    USING_GPU = True
except ImportError:
    print("CuPy 不可用，NumPy使用CPU计算")
    import numpy as np
    USING_GPU = False

import librosa
import matplotlib.pyplot as plt
import soundfile as sf

CuPy (GPU) 可用: 12090
GPU设备: <CUDA Device 0>


## 1. 加载音频文件

使用 `librosa.load()` 函数加载音频文件。设置 `mono=False` 以保留原始声道信息。

In [34]:
def ProccessFrequency(left_fft, right_fft) -> tuple[float, float]:
    left_fft = np.fft.fft(left_fft)
    right_fft = np.fft.fft(right_fft)
    left_fft *= np.conj(left_fft)
    right_fft *= np.conj(right_fft)
    left_fft = np.real(left_fft)
    right_fft = np.real(right_fft)
    left_fft = np.sqrt(left_fft)
    right_fft = np.sqrt(right_fft)
    left_sum = np.sum(left_fft[1:])
    right_sum = np.sum(right_fft[1:])
    if left_sum > right_sum :
        return (1, left_sum / right_sum)
    else :
        return (right_sum / left_sum, 1)




In [35]:
# 处理单声道音频
def make_sound_blence(audio_path, output_path):
    y, sr = librosa.load(audio_path, sr=None, mono=False)
    y = np.array(y)
    if y.ndim == 1:
        y = np.stack([y, y], axis=0)
        sf.write(output_path, y.T, sr)  # soundfile 需要转置数组 (channels, samples) -> (samples, channels)
        return

    # 分离左右声道
    l, r = ProccessFrequency(y[0], y[1])
    y[0] *= l
    y[1] *= r
    # final_y = (y[0] + y[1]) * 0.5
    # final_y = np.stack([final_y, final_y], axis=0)
    sf.write(output_path, final_y.T.get(), sr)  # soundfile 需要转置数组 (channels, samples) -> (samples, channels)


In [36]:
# 处理所有图片
from pathlib import Path
source_dir = Path("Item_Origin")
output_dir = Path("Item")
image_files = list(source_dir.rglob("*.wav"))
print(f"找到 {len(image_files)} 段音频")

for img_path in image_files:
    # 保持目录结构输出
    rel_path = img_path.relative_to(source_dir)
    output_path = output_dir / rel_path
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    make_sound_blence(img_path, output_path)
    print(f"已处理: {rel_path}")

print(f"\n完成! 共处理 {len(image_files)} 段音频")

找到 64 段音频


NameError: name 'final_y' is not defined